# 01 — Construcción del corpus normativo

Este notebook inicializa y audita el registro de instrumentos oficiales. No descarga documentos automáticamente y no modifica los archivos originales.

**Objetivo:** construir un corpus reproducible de normativa, políticas y planes de manejo sobre cambio climático, variabilidad ambiental y pesquerías, con énfasis en pequeños pelágicos y aplicabilidad a la anchoveta peruana.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import yaml

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from regulatory_review.schema import ScreeningDecision, RegulatoryRecord
from regulatory_review.scoring import score_transferability

print(f'Project root: {ROOT}')

## 1. Cargar la taxonomía controlada

In [ ]:
taxonomy_path = ROOT / 'config' / 'taxonomy.yml'
with taxonomy_path.open('r', encoding='utf-8') as stream:
    taxonomy = yaml.safe_load(stream)

print(taxonomy['project']['name'])
print('Management measures:', len(taxonomy['management_measures']))
print('Trigger types:', len(taxonomy['trigger_types']))

## 2. Abrir el registro de fuentes oficiales

Cada documento debe registrarse antes de ser procesado. La plantilla se mantiene vacía en GitHub; las filas de trabajo se agregan localmente o en una copia controlada.

In [ ]:
registry_path = ROOT / 'data' / 'templates' / 'source_registry.csv'
registry = pd.read_csv(registry_path)
registry.info()
registry.head()

## 3. Validar estructura y duplicados

In [ ]:
required_columns = {
    'document_id', 'title', 'jurisdiction', 'year', 'instrument_type',
    'legal_status', 'competent_authority', 'official_url',
    'access_date', 'source_language', 'local_path'
}

missing_columns = required_columns.difference(registry.columns)
if missing_columns:
    raise ValueError(f'Missing registry columns: {sorted(missing_columns)}')

if registry['document_id'].dropna().duplicated().any():
    duplicates = registry.loc[registry['document_id'].duplicated(False), 'document_id']
    raise ValueError(f'Duplicate document_id values: {duplicates.tolist()}')

print(f'Registered documents: {len(registry):,}')
print('Registry structure is valid.')

## 4. Crear una fila nueva sin escribirla automáticamente

Complete este diccionario únicamente con información verificable de la fuente oficial. La celda genera una fila candidata para revisión.

In [ ]:
candidate_source = {
    'document_id': 'jurisdiction_year_short_title',
    'title': '',
    'jurisdiction': '',
    'region': '',
    'year': None,
    'version_date': '',
    'instrument_type': '',
    'legal_status': 'unclear',
    'competent_authority': '',
    'official_url': '',
    'access_date': '',
    'source_language': '',
    'local_path': '',
    'checksum_sha256': '',
    'notes': ''
}

pd.DataFrame([candidate_source])

## 5. Comprobar la puntuación de transferibilidad

La puntuación se usa para priorizar revisión, no como conclusión jurídica automática.

In [ ]:
example_score = score_transferability(
    ecological_similarity=3,
    climate_operationalisation=2,
    trigger_specificity=2,
    response_predefinition=2,
    legal_force=2,
    data_feasibility_peru=2,
)
example_score

## Próximos pasos

1. Construir la estrategia de búsqueda y el corpus semilla.
2. Descargar y versionar documentos oficiales.
3. Extraer texto por artículo o sección.
4. Implementar cribado con alta sensibilidad.
5. Implementar RAG y extracción JSON.
6. Validar manualmente todas las inclusiones y afirmaciones vinculantes.